# SVTRv2 Fine-tuning on Custom Data

This notebook fine-tunes SVTRv2 (SMTR+GTC+RCTC) on your custom OCR dataset.

**Steps:**
1. Mount Google Drive
2. Clone OpenOCR
3. Install dependencies
4. Upload custom data
5. Download pretrained checkpoint
6. Create fine-tune config
7. Train
8. Evaluate

> Make sure **Runtime > Change runtime type > GPU (T4)** is selected.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Check GPU

In [ ]:
!nvidia-smi

Wed Jun 24 13:51:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Clone OpenOCR

In [ ]:
%cd /content
!git clone https://github.com/ngocthien2306/OpenOCR.git
%cd /content/OpenOCR

/content
Cloning into 'OpenOCR'...
remote: Enumerating objects: 2923, done.
remote: Counting objects: 100% (1536/1536), done.
remote: Compressing objects: 100% (361/361), done.
remote: Total 2923 (delta 1258), reused 1175 (delta 1175), pack-reused 1387 (from 1)
Receiving objects: 100% (2923/2923), 3.56 MiB | 16.59 MiB/s, done.
Resolving deltas: 100% (1926/1926), done.
/content/OpenOCR


## 4. Install Dependencies

In [ ]:
!pip install -r requirements.txt -q
!pip install lmdb tqdm -q
print('Done!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 121.8 MB/s eta 0:00:00
Done!


## 5. Upload Custom Data

**Option A — Upload a zip from your computer**

Prepare a zip file with this structure:
```
data_ocr.zip
  data_ocr/
    train/
      img1.jpg
      img2.jpg
    test/
      img3.jpg
    rec_gt_train.txt    # format: train/img1.jpg<TAB>label
    rec_gt_test.txt
```

In [ ]:
# Option A: Upload zip from your computer
from google.colab import files
uploaded = files.upload()  # select your data_ocr.zip

Saving data_ocr_merged.zip to data_ocr_merged.zip


In [ ]:
# Unzip uploaded file
import os
zip_name = 'data_ocr.zip'
!unzip -q "{zip_name}" -d /content/OpenOCR/
print('Extracted to /content/OpenOCR/data_ocr')

Extracted to /content/OpenOCR/data_ocr


In [ ]:
# Option B: Copy from Google Drive (if you already uploaded there)
# !cp -r /content/drive/MyDrive/data_ocr /content/OpenOCR/data_ocr

In [ ]:
# Verify data
import os
data_dir = '/content/OpenOCR/data_ocr'

train_txt = os.path.join(data_dir, 'rec_gt_train.txt')
test_txt  = os.path.join(data_dir, 'rec_gt_test.txt')

with open(train_txt) as f:
    train_lines = f.readlines()
with open(test_txt) as f:
    test_lines = f.readlines()

print(f'Train samples : {len(train_lines)}')
print(f'Test  samples : {len(test_lines)}')
print(f'\nSample lines:')
for l in train_lines[:3]:
    print(' ', l.strip())

Train samples : 640
Test  samples : 156

Sample lines:
  train/69b99ada432661a42ba94d3b_40767174_f0_ann2__8B_MA__exp__BB_MA__vc__0.8929__thr__0.60.jpg	BB/MA
  train/6986a77dd15b01fe139ff0d3_40767174_f0_ann2__BB_MA 2029 FE 06__exp__BB_MA 2029 FE 06__vc__-1.0000__thr__-1.00.jpg	BB/MA 2029 FE 06
  train/698ece5124cf1f27b970c178_40767174_f0_ann3__2029.02.12__exp__12_02_2029__vc__0.9446__thr__0.70.jpg	2029.02.12


## 6. Download Pretrained Checkpoint

## 7. Create Fine-tune Config

In [ ]:
import os

ckpt_path = "/content/drive/MyDrive/svtrv2_finetune_output_18_6/best.pth"
# ========== SETTINGS — edit these ==========
CHECKPOINT_PATH = ckpt_path
DATA_DIR        = '/content/OpenOCR/data_ocr'
OUTPUT_DIR      = '/content/OpenOCR/output/finetune'
EPOCH_NUM       = 50
BATCH_SIZE      = 32
LR              = 0.0001
USE_SPACE_CHAR  = False
MAX_TEXT_LEN    = 25
IMAGE_H         = 32
IMAGE_W         = 128
# ===========================================

config_content = f"""Global:
  device: gpu
  epoch_num: {EPOCH_NUM}
  log_smooth_window: 20
  print_batch_step: 10
  output_dir: {OUTPUT_DIR}
  save_epoch_step: [1, 1]
  eval_batch_step: [0, 500]
  eval_epoch_step: [0, 1]
  cal_metric_during_train: True
  pretrained_model: {CHECKPOINT_PATH}
  checkpoints:
  use_tensorboard: false
  infer_img:
  character_dict_path: &character_dict_path ./tools/utils/EN_symbol_dict.txt
  max_text_length: &max_text_length {MAX_TEXT_LEN}
  use_space_char: &use_space_char {USE_SPACE_CHAR}
  save_res_path: {OUTPUT_DIR}/predicts.txt
  use_amp: True

Optimizer:
  name: AdamW
  lr: {LR}
  weight_decay: 0.05
  filter_bias_and_bn: True

LRScheduler:
  name: OneCycleLR
  warmup_epoch: 1.5
  cycle_momentum: False

Architecture:
  model_type: rec
  algorithm: SVTRv2
  in_channels: 3
  Transform:
  Encoder:
    name: SVTRv2LNConvTwo33
    use_pos_embed: False
    dims: [128, 256, 384]
    depths: [6, 6, 6]
    num_heads: [4, 8, 12]
    mixer: [['Conv','Conv','Conv','Conv','Conv','Conv'],['Conv','Conv','FGlobal','Global','Global','Global'],['Global','Global','Global','Global','Global','Global']]
    local_k: [[5, 5], [5, 5], [-1, -1]]
    sub_k: [[1, 1], [2, 1], [-1, -1]]
    last_stage: false
    feat2d: True
  Decoder:
    name: GTCDecoder
    infer_gtc: True
    detach: False
    gtc_decoder:
      name: SMTRDecoder
      num_layer: 1
      ds: True
      max_len: *max_text_length
      next_mode: &next True
      sub_str_len: &subsl 5
    ctc_decoder:
      name: RCTCDecoder

Loss:
  name: GTCLoss
  ctc_weight: 0.1
  gtc_loss:
    name: SMTRLoss

PostProcess:
  name: GTCLabelDecode
  gtc_label_decode:
    name: SMTRLabelDecode
    next_mode: *next
  character_dict_path: *character_dict_path
  use_space_char: *use_space_char

Metric:
  name: RecGTCMetric
  main_indicator: acc
  is_filter: True

Train:
  dataset:
    name: SimpleDataSet
    data_dir: {DATA_DIR}
    label_file_list: [\"{DATA_DIR}/rec_gt_train.txt\"]
    transforms:
      - DecodeImagePIL:
          img_mode: RGB
      - PARSeqAugPIL:
      - RecTVResize:
          image_shape: [{IMAGE_H}, {IMAGE_W}]
          padding: False
      - GTCLabelEncode:
          gtc_label_encode:
            name: SMTRLabelEncode
            sub_str_len: *subsl
          character_dict_path: *character_dict_path
          use_space_char: *use_space_char
          max_text_length: *max_text_length
      - KeepKeys:
          keep_keys: ['image', 'label', 'label_subs', 'label_next', 'length_subs',
          'label_subs_pre', 'label_next_pre', 'length_subs_pre', 'length', 'ctc_label', 'ctc_length']
  loader:
    shuffle: True
    batch_size_per_card: {BATCH_SIZE}
    drop_last: True
    num_workers: 2

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: {DATA_DIR}
    label_file_list: [\"{DATA_DIR}/rec_gt_test.txt\"]
    transforms:
      - DecodeImagePIL:
          img_mode: RGB
      - RecTVResize:
          image_shape: [{IMAGE_H}, {IMAGE_W}]
          padding: False
      - GTCLabelEncode:
          gtc_label_encode:
            name: ARLabelEncode
          character_dict_path: *character_dict_path
          use_space_char: *use_space_char
          max_text_length: *max_text_length
      - KeepKeys:
          keep_keys: ['image', 'label', 'length', 'ctc_label', 'ctc_length']
  loader:
    shuffle: False
    drop_last: False
    batch_size_per_card: {BATCH_SIZE}
    num_workers: 2
"""

config_path = '/content/OpenOCR/configs/rec/svtrv2/svtrv2_finetune_custom.yml'
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, 'w') as f:
    f.write(config_content)

print(f'Config saved to : {config_path}')

Config saved to : /content/OpenOCR/configs/rec/svtrv2/svtrv2_finetune_custom.yml


In [ ]:
!pip install numpy==1.26.4 --upgrade


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >=

In [ ]:
!grep -n "class SVTRResize" /content/OpenOCR/openrec/preprocess/*.py -A 30

/content/OpenOCR/openrec/preprocess/resize.py:65:class SVTRResize(object):
/content/OpenOCR/openrec/preprocess/resize.py-66-
/content/OpenOCR/openrec/preprocess/resize.py-67-    def __init__(self, image_shape, padding=True, **kwargs):
/content/OpenOCR/openrec/preprocess/resize.py-68-        self.image_shape = image_shape
/content/OpenOCR/openrec/preprocess/resize.py-69-        self.padding = padding
/content/OpenOCR/openrec/preprocess/resize.py-70-
/content/OpenOCR/openrec/preprocess/resize.py-71-    def __call__(self, data):
/content/OpenOCR/openrec/preprocess/resize.py-72-        img = data['image']
/content/OpenOCR/openrec/preprocess/resize.py-73-        h, w = img.shape[:2]
/content/OpenOCR/openrec/preprocess/resize.py-74-        norm_img, valid_ratio = resize_norm_img(img, self.image_shape,
/content/OpenOCR/openrec/preprocess/resize.py-75-                                                self.padding)
/content/OpenOCR/openrec/preprocess/resize.py-76-        data['image'] = norm_img


## 8. Start Fine-tuning

In [ ]:
!sed -i 's/preds = preds\.detach()\.cpu()\.numpy()/preds = preds.detach().cpu().float().numpy()/g' \
  /content/OpenOCR/openrec/postprocess/smtr_postprocess.py

In [ ]:
%cd /content/OpenOCR

# Patch ctc_postprocess.py to handle BFloat16 conversion
!sed -i 's/preds = preds\.detach()\.cpu()\.numpy()/preds = preds.detach().cpu().float().numpy()/g' \
  /content/OpenOCR/openrec/postprocess/ctc_postprocess.py

!CUDA_VISIBLE_DEVICES=0 torchrun --nproc_per_node=1 \
    tools/train_rec.py \
    --c configs/rec/svtrv2/svtrv2_finetune_custom.yml

/content/OpenOCR
[2026/06/24 14:52:43] openrec INFO: ----------- Config -----------
[2026/06/24 14:52:43] openrec INFO: Architecture : 
[2026/06/24 14:52:43] openrec INFO:     Decoder : 
[2026/06/24 14:52:43] openrec INFO:         ctc_decoder : 
[2026/06/24 14:52:43] openrec INFO:             name : RCTCDecoder
[2026/06/24 14:52:43] openrec INFO:         detach : False
[2026/06/24 14:52:43] openrec INFO:         gtc_decoder : 
[2026/06/24 14:52:43] openrec INFO:             ds : True
[2026/06/24 14:52:43] openrec INFO:             max_len : 25
[2026/06/24 14:52:43] openrec INFO:             name : SMTRDecoder
[2026/06/24 14:52:43] openrec INFO:             next_mode : True
[2026/06/24 14:52:43] openrec INFO:             num_layer : 1
[2026/06/24 14:52:43] openrec INFO:             sub_str_len : 5
[2026/06/24 14:52:43] openrec INFO:         infer_gtc : True
[2026/06/24 14:52:43] openrec INFO:         name : GTCDecoder
[2026/06/24 14:52:43] openrec INFO:     Encoder : 
[2026/06/24 14:52:

In [ ]:
import os

ctc_postprocess_path = '/content/OpenOCR/openrec/postprocess/ctc_postprocess.py'

with open(ctc_postprocess_path, 'r') as f:
    content = f.read()

old_code = 'preds = preds.detach().cpu().numpy()'
new_code = 'preds = preds.detach().cpu().float().numpy()'

if old_code in content:
    content = content.replace(old_code, new_code)
    with open(ctc_postprocess_path, 'w') as f:
        f.write(content)
    print('Successfully patched ctc_postprocess.py to handle BFloat16 conversion.')
else:
    print('ctc_postprocess.py already patched or target line not found.')

ctc_postprocess.py already patched or target line not found.


The `ctc_postprocess.py` has been patched. Please re-run the training cell below (cell `7gRN4cHdWOPS`).

## 9. Evaluate Best Model

In [ ]:
%cd /content/OpenOCR
best_ckpt = '/content/OpenOCR/output/finetune/best.pth'

!python tools/eval_rec.py \
    --c configs/rec/svtrv2/svtrv2_finetune_custom.yml \
    --o Global.pretrained_model={best_ckpt}

/content/OpenOCR
[2026/06/17 16:39:34] openrec INFO: ----------- Config -----------
[2026/06/17 16:39:34] openrec INFO: Architecture : 
[2026/06/17 16:39:34] openrec INFO:     Decoder : 
[2026/06/17 16:39:34] openrec INFO:         ctc_decoder : 
[2026/06/17 16:39:34] openrec INFO:             name : RCTCDecoder
[2026/06/17 16:39:34] openrec INFO:         detach : False
[2026/06/17 16:39:34] openrec INFO:         gtc_decoder : 
[2026/06/17 16:39:34] openrec INFO:             ds : True
[2026/06/17 16:39:34] openrec INFO:             max_len : 25
[2026/06/17 16:39:34] openrec INFO:             name : SMTRDecoder
[2026/06/17 16:39:34] openrec INFO:             next_mode : True
[2026/06/17 16:39:34] openrec INFO:             num_layer : 1
[2026/06/17 16:39:34] openrec INFO:             sub_str_len : 5
[2026/06/17 16:39:34] openrec INFO:         infer_gtc : True
[2026/06/17 16:39:34] openrec INFO:         name : GTCDecoder
[2026/06/17 16:39:34] openrec INFO:     Encoder : 
[2026/06/17 16:39: